# 1. Obtención de Datos (Get Data)

**Objetivo del notebook:** descargar el dataset de origen desde Kaggle y dejarlo disponible en la carpeta `data/` del proyecto, listo para ser consumido por los siguientes módulos (`2_EDA.ipynb`, `3_FeatureEngineering.ipynb` y `4_Modeling.ipynb`).

### Estructura del proyecto
```
proyecto/
├── data/        # datasets: crudo y preprocesado
├── models/      # modelos entrenados y artefactos serializados (.pkl)
├── notebooks/   # los 4 notebooks del módulo (este directorio)
└── output/      # resultados exportados (ej. Excel con predicciones)
```
Cada notebook lee y escribe siempre sobre estas mismas carpetas relativas (`../data/`, `../models/`, `../output/`), por lo que deben ejecutarse desde `notebooks/`.

### Caso de negocio: Dataset Telco Customer Churn

### ¿Qué es el “churn”?
En telecomunicaciones, **churn** (cancelación, deserción o abandono) es el evento en el cual un cliente deja de utilizar el servicio de la compañía.

El problema es crítico porque:
* Conseguir nuevos clientes suele ser más costoso que retener los actuales.
* La pérdida de clientes afecta ingresos recurrentes.
* Permite identificar segmentos con riesgo de abandono.
* Facilita campañas de retención, fidelización y optimización comercial.

### Objetivo del dataset
El dataset fue diseñado para estudiar el problema de: 
~~~cmd
Predecir si un cliente de telecomunicaciones abandonará la compañía (Churn = Yes) o permanecerá activo (Churn = No)
~~~
Sin embargo, antes del modelado, el dataset permite trabajar:
* Entendimiento del negocio.
* Tipología de clientes.
* Productos contratados.
* Comportamiento de facturación.
* Permanencia.
* Métodos de pago.
* Riesgo de abandono.

**Unidad de análisis**: cliente de la empresa; cada fila representa un cliente.


### Variables del dataset
| Columna | Descripción | Tipo de dato |
|---|---|---|
| `customerID` | Identificador único del cliente. | Texto / Categórica identificadora |
| `gender` | Género del cliente. | Categórica nominal |
| `SeniorCitizen` | Indica si el cliente es adulto mayor (1 = Sí, 0 = No). | Binaria (entero categórico) |
| `Partner` | Indica si el cliente tiene pareja o cónyuge. | Categórica binaria |
| `Dependents` | Indica si el cliente tiene personas dependientes económicamente. | Categórica binaria |
| `tenure` | Número de meses que el cliente ha permanecido con la compañía. | Numérica discreta (entero) |
| `PhoneService` | Indica si el cliente tiene servicio telefónico. | Categórica binaria |
| `MultipleLines` | Indica si el cliente tiene múltiples líneas telefónicas. | Categórica nominal |
| `InternetService` | Tipo de servicio de internet contratado. | Categórica nominal |
| `OnlineSecurity` | Indica si el cliente tiene servicio de seguridad en línea. | Categórica nominal |
| `OnlineBackup` | Indica si el cliente tiene servicio de respaldo en línea. | Categórica nominal |
| `DeviceProtection` | Indica si el cliente tiene protección para dispositivos. | Categórica nominal |
| `TechSupport` | Indica si el cliente tiene soporte técnico contratado. | Categórica nominal |
| `StreamingTV` | Indica si el cliente tiene servicio de televisión por streaming. | Categórica nominal |
| `StreamingMovies` | Indica si el cliente tiene servicio de películas por streaming. | Categórica nominal |
| `Contract` | Tipo de contrato del cliente (mensual, anual o bianual). | Categórica nominal (con orden conceptual) |
| `PaperlessBilling` | Indica si el cliente recibe facturación electrónica. | Categórica binaria |
| `PaymentMethod` | Método de pago utilizado por el cliente. | Categórica nominal |
| `MonthlyCharges` | Valor de facturación mensual actual del cliente. | Numérica continua (decimal) |
| `TotalCharges` | Valor total facturado al cliente durante toda su relación con la compañía. | Numérica continua (decimal) |
| `Churn` | Indica si el cliente abandonó la compañía. | Categórica binaria (variable objetivo) |


### Configuración inicial

Antes de instalar o importar librerías, se valida cuál es el intérprete de Python (kernel) que está usando el notebook. Esto evita errores comunes en clase cuando el kernel activo no corresponde al entorno virtual donde están instaladas las dependencias del proyecto (por ejemplo `kagglehub`).

In [1]:
## validación del Kernel seleccionado
import sys
print(sys.executable)

c:\Python\venvs\py312\Scripts\python.exe


Se importan las librerías necesarias:
- `os`, `glob`, `shutil`: manejo de archivos y rutas del sistema operativo.
- `kagglehub`: cliente oficial para descargar datasets directamente desde Kaggle sin necesidad de   descargarlos manualmente desde el navegador.
- `pandas`: manipulación de datos tabulares.

In [ ]:
import os, glob, shutil##, kagglehub

import pandas as pd
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

### Verificar / crear la estructura de carpetas del proyecto

Este es el primer notebook que escribe archivos en disco (copia el dataset descargado a `../data/`). Antes de hacerlo, se valida que existan las carpetas que el proyecto completo necesita — `data/`, `models/` y `output/` — y se crean automáticamente si faltan. Esto hace que el proyecto sea reproducible: cualquier persona que clone el repositorio y ejecute los notebooks en orden, desde una carpeta vacía, no tendrá errores por carpetas inexistentes en `4_Modeling.ipynb` (que guarda modelos en `../models/` y resultados en `../output/`).

In [3]:
carpetas_requeridas = ["../data", "../models", "../output"]

for carpeta in carpetas_requeridas:
    os.makedirs(carpeta, exist_ok=True)

print("Carpetas verificadas/creadas:", carpetas_requeridas)

Carpetas verificadas/creadas: ['../data', '../models', '../output']


## Obtener datos de origen

Se descarga el dataset [telco-customer-churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) usando `kagglehub`, que se encarga de autenticarse con la API de Kaggle (requiere tener configuradas las credenciales de Kaggle en el entorno) y descargar la versión más reciente del dataset a una carpeta caché local. Luego se copian los archivos descargados a `../data/`, para que el resto del proyecto los referencie siempre desde una misma ubicación relativa, sin depender de la ruta caché de Kaggle.

In [ ]:
# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")
files = [f.replace("\\","/") for f in glob.glob( path.replace("\\","/") + "/*", recursive=True) if os.path.isfile(f)]
resp = [shutil.copy(f, "../data/" + os.path.basename(f)) for f in files]
fName = resp[0]
print("Path to dataset files:", resp[0])
print(f"Archivos descargados: {len(resp)}")
print("\t* "+ "\n\t* ".join(resp))

### Alternativa: cargar el archivo directamente desde `data/`

Si no tienes configuradas las credenciales de la API de Kaggle (`kaggle.json` o las variables de entorno `KAGGLE_USERNAME` / `KAGGLE_KEY`), la celda de `kagglehub` anterior fallará. En ese caso, descarga el archivo manualmente desde la [página del dataset en Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) (botón *Download*), colócalo sin cambiarle el nombre en la carpeta `../data/` de este proyecto, y ejecuta esta celda en lugar de la anterior:

In [4]:
# Alternativa sin kagglehub: usar directamente el archivo ya ubicado en ../data/
fName = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
print("Archivo esperado en:", fName)

Archivo esperado en: ../data/WA_Fn-UseC_-Telco-Customer-Churn.csv


## Cargar datos

Se lee el archivo con `pandas`, sin ninguna transformación adicional, usando la variable `fName` definida en la celda de `kagglehub` o en la alternativa manual anterior. Este es el dataset **crudo**: la limpieza y transformación se abordan en los notebooks siguientes (`2_EDA.ipynb` y `3_FeatureEngineering.ipynb`).

In [5]:
df = pd.read_csv(fName, sep=",")
print("Dimensiones de la tabla:", df.shape)
df.head()

Dimensiones de la tabla: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Se revisan los tipos de datos asignados automáticamente por `pandas` al importar el CSV. Esta es la primera señal de posibles inconsistencias (por ejemplo, columnas numéricas que quedan como texto), que se investigarán a fondo en el módulo de EDA.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

### Siguiente paso

El dataset queda descargado en `../data/` en su versión original (sin modificar). El notebook `2_EDA.ipynb` parte de este mismo archivo para realizar el análisis exploratorio.